# Build Texas ZCTA ACS Target Profiles

This notebook builds one ACS target-profile row per Texas ZCTA. The output is used later to control demographic sampling from PUMS donor profiles: ACS targets preserve local ZIP/ZCTA distributions, while PUMS later preserves realistic combinations of age, education, income, household structure, occupation, and homeownership.

This notebook does not create synthetic customers and does not pull PUMS data.

## 1. Imports and Configuration

Paths are built with `pathlib`. The notebook is restartable and uses cached raw ACS files unless `FORCE_REFRESH` is set to `True`.

In [1]:
from pathlib import Path
import json
import math
import os
import time

import numpy as np
import pandas as pd
import requests

PROJECT_ROOT = Path("Synthetic MySQL Tables") / "Customers"
if not PROJECT_ROOT.exists() and Path.cwd().name == "Customers":
    PROJECT_ROOT = Path(".")

RAW_DIR = PROJECT_ROOT / "raw" / "census"
PROCESSED_DIR = PROJECT_ROOT / "processed" / "census"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ACS_YEAR = 2024
ACS_DATASET = "acs/acs5"
STATE_FIPS = "48"
BASE_API_URL = f"https://api.census.gov/data/{ACS_YEAR}/{ACS_DATASET}"
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
FORCE_REFRESH = False

OUTPUT_MAIN = PROCESSED_DIR / "tx_zcta_acs_targets_2024.csv"
OUTPUT_AUDIT = PROCESSED_DIR / "tx_zcta_acs_targets_2024_audit.csv"
OUTPUT_MISSING_ZCTAS = PROCESSED_DIR / "missing_zctas_from_acs_2024.csv"

ZIP_COLUMN_CANDIDATES = ["zcta", "zip_code", "zip", "assigned_zip", "customer_zip_code", "ZIP", "ZCTA"]
TEXAS_ZCTA_PREFIXES = ("75", "76", "77", "78", "79", "88")

TARGET_ZCTA_FILE_CANDIDATES = [
    PROJECT_ROOT / "processed" / "census" / "tx_zcta_to_puma_crosswalk_2020.csv",
    PROJECT_ROOT / "tx_zcta_to_puma_crosswalk_2020.csv",
    PROJECT_ROOT / "texas_target_weights.csv",
    PROJECT_ROOT / "postal_customers.csv",
    PROJECT_ROOT / "data" / "processed" / "census" / "tx_zcta_to_puma_crosswalk_2020.csv",
    Path("Synthetic_Customers_and_Orders") / "postal_customers.csv",
    Path("Establish_source_codes_and_target_weights") / "texas_target_weights.csv",
]

print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Raw ACS cache: {RAW_DIR.resolve()}")
print(f"Processed output: {PROCESSED_DIR.resolve()}")
print(f"Census API key available: {bool(CENSUS_API_KEY)}")

Project root: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Customers
Raw ACS cache: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Customers\raw\census
Processed output: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Customers\processed\census
Census API key available: True


## 2. Define ACS Variables

The variables below come from the 2024 ACS 5-year Detailed Tables API. They are grouped by source table so each raw cache file remains easy to audit.

In [2]:
ACS_VARIABLE_GROUPS = {
    "B19001": {
        "description": "Household income in the past 12 months",
        "variables": [f"B19001_{i:03d}E" for i in range(1, 18)],
    },
    "B25003": {
        "description": "Tenure",
        "variables": ["B25003_001E", "B25003_002E", "B25003_003E"],
    },
    "B11001": {
        "description": "Household type",
        "variables": ["B11001_001E", "B11001_002E", "B11001_003E"],
    },
    "B15003": {
        "description": "Educational attainment for population 25 years and over",
        "variables": [f"B15003_{i:03d}E" for i in range(1, 26)],
    },
    "B01001": {
        "description": "Sex by age",
        "variables": [
            *[f"B01001_{i:03d}E" for i in range(7, 26)],
            *[f"B01001_{i:03d}E" for i in range(31, 50)],
        ],
    },
}

ALL_ACS_VARIABLES = sorted({variable for group in ACS_VARIABLE_GROUPS.values() for variable in group["variables"]})
print(f"ACS variable groups: {list(ACS_VARIABLE_GROUPS)}")
print(f"Unique ACS estimate variables: {len(ALL_ACS_VARIABLES):,}")

ACS variable groups: ['B19001', 'B25003', 'B11001', 'B15003', 'B01001']
Unique ACS estimate variables: 86


## 3. Pull ACS Data

Each table is cached as a raw CSV. If the Census API request list ever gets too long, the fetch helper splits variables into chunks and merges them back on geography.

In [3]:
def chunked(values: list[str], size: int) -> list[list[str]]:
    return [values[index:index + size] for index in range(0, len(values), size)]


def fetch_acs_json(params: dict, retries: int = 4, backoff_seconds: float = 1.5) -> list[list[str]]:
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(BASE_API_URL, params=params, timeout=60)
            response.raise_for_status()
            if "missing_key.html" in response.url:
                raise RuntimeError("The Census API requested a valid key for this endpoint. Set CENSUS_API_KEY and rerun, or use cached raw ACS files.")
            if not response.text.lstrip().startswith("["):
                excerpt = response.text[:300].replace("\n", " ")
                raise RuntimeError(f"ACS API returned a non-JSON response from {response.url}: {excerpt}")
            return response.json()
        except Exception as error:
            last_error = error
            wait_seconds = backoff_seconds * attempt
            print(f"ACS request failed on attempt {attempt}/{retries}: {error}. Retrying in {wait_seconds:.1f}s")
            time.sleep(wait_seconds)
    raise RuntimeError(f"ACS request failed after {retries} attempts") from last_error


def acs_json_to_frame(payload: list[list[str]]) -> pd.DataFrame:
    if not payload or len(payload) < 2:
        raise ValueError("ACS API returned no data rows")
    return pd.DataFrame(payload[1:], columns=payload[0])


def fetch_acs_table(year: int, dataset: str, variables: list[str], geography: str, api_key: str | None = None) -> pd.DataFrame:
    del year, dataset
    geog_column = geography.split(":", 1)[0]
    frames = []

    for variable_chunk in chunked(variables, 45):
        params = {
            "get": ",".join(["NAME", *variable_chunk]),
            "for": geography,
        }
        if api_key:
            params["key"] = api_key

        payload = fetch_acs_json(params)
        frames.append(acs_json_to_frame(payload))

    merged = frames[0]
    for frame in frames[1:]:
        merged = merged.merge(frame, on=["NAME", geog_column], how="outer")
    return merged


def load_or_fetch_group(group_name: str, variables: list[str]) -> pd.DataFrame:
    cache_path = RAW_DIR / f"acs5_{ACS_YEAR}_{group_name}_zcta_raw.csv"
    if cache_path.exists() and not FORCE_REFRESH:
        print(f"Loading cached {group_name}: {cache_path}")
        return pd.read_csv(cache_path, dtype="string", low_memory=False)

    print(f"Fetching {group_name} from ACS API")
    frame = fetch_acs_table(
        year=ACS_YEAR,
        dataset=ACS_DATASET,
        variables=variables,
        geography="zip code tabulation area:*",
        api_key=CENSUS_API_KEY,
    )
    frame.to_csv(cache_path, index=False)
    print(f"Cached {group_name}: {cache_path} ({len(frame):,} rows)")
    return frame


raw_acs_tables = {
    group_name: load_or_fetch_group(group_name, group_config["variables"])
    for group_name, group_config in ACS_VARIABLE_GROUPS.items()
}

Fetching B19001 from ACS API


Cached B19001: raw\census\acs5_2024_B19001_zcta_raw.csv (33,772 rows)
Fetching B25003 from ACS API


Cached B25003: raw\census\acs5_2024_B25003_zcta_raw.csv (33,772 rows)
Fetching B11001 from ACS API


Cached B11001: raw\census\acs5_2024_B11001_zcta_raw.csv (33,772 rows)
Fetching B15003 from ACS API


Cached B15003: raw\census\acs5_2024_B15003_zcta_raw.csv (33,772 rows)
Fetching B01001 from ACS API


Cached B01001: raw\census\acs5_2024_B01001_zcta_raw.csv (33,772 rows)


## 4. Normalize and Merge ACS Tables

The ACS geography field is normalized to `zcta`, and all estimate columns are converted to numeric values before target calculations.

In [4]:
def normalize_acs_table(frame: pd.DataFrame) -> pd.DataFrame:
    normalized = frame.copy()
    geog_columns = [column for column in normalized.columns if column.lower() == "zip code tabulation area"]
    if not geog_columns:
        raise ValueError("Expected ACS geography column 'zip code tabulation area'")

    geog_column = geog_columns[0]
    normalized["zcta"] = normalized[geog_column].astype("string").str.strip().str.zfill(5)
    normalized = normalized.drop(columns=[geog_column])
    return normalized


normalized_tables = {group_name: normalize_acs_table(frame) for group_name, frame in raw_acs_tables.items()}

acs_merged = None
for group_name, frame in normalized_tables.items():
    if acs_merged is None:
        acs_merged = frame.copy()
    else:
        acs_merged = acs_merged.merge(frame, on=["zcta", "NAME"], how="outer")
    print(f"Merged {group_name}: {len(acs_merged):,} rows")

acs_merged = acs_merged.rename(columns={"NAME": "zcta_name"})
for column in ALL_ACS_VARIABLES:
    if column in acs_merged.columns:
        acs_merged[column] = pd.to_numeric(acs_merged[column], errors="coerce")

print(f"Merged ACS ZCTA rows before Texas filtering: {len(acs_merged):,}")

Merged B19001: 33,772 rows


Merged B25003: 33,772 rows
Merged B11001: 33,772 rows


Merged B15003: 33,772 rows


Merged B01001: 33,772 rows


Merged ACS ZCTA rows before Texas filtering: 33,772


## 5. Filter to Texas Target ZCTAs

When available, a local crosswalk or customer file defines the target ZCTA list. If no local list exists, the notebook falls back to Texas-like ZCTA prefixes (`75`, `76`, `77`, `78`, `79`, `88`). Prefix filtering is only a fallback; the crosswalk or target ZIP list is preferred.

In [5]:
def infer_zip_column(columns: pd.Index) -> str | None:
    exact_lookup = {column.lower(): column for column in columns}
    for candidate in ZIP_COLUMN_CANDIDATES:
        if candidate.lower() in exact_lookup:
            return exact_lookup[candidate.lower()]
    return None


def load_target_zctas() -> tuple[set[str] | None, Path | None, str | None]:
    for candidate_path in TARGET_ZCTA_FILE_CANDIDATES:
        if not candidate_path.exists():
            continue

        target_frame = pd.read_csv(candidate_path, dtype="string", low_memory=False)
        zip_column = "zcta" if "zcta" in target_frame.columns else infer_zip_column(target_frame.columns)
        if zip_column is None:
            print(f"Found {candidate_path}, but no ZIP/ZCTA column was recognized.")
            continue

        target_zctas = set(target_frame[zip_column].dropna().astype("string").str.strip().str.zfill(5))
        print(f"Using target ZCTA list from: {candidate_path}")
        print(f"Inferred ZIP/ZCTA column: {zip_column}")
        print(f"Target ZCTA count: {len(target_zctas):,}")
        return target_zctas, candidate_path, zip_column

    print("No local target ZCTA file found. Falling back to Texas-like ZCTA prefixes.")
    return None, None, None


target_zctas, target_zcta_path, target_zcta_column = load_target_zctas()

if target_zctas is None:
    filtered_acs = acs_merged.loc[acs_merged["zcta"].str.startswith(TEXAS_ZCTA_PREFIXES, na=False)].copy()
    used_target_file = False
else:
    filtered_acs = acs_merged.loc[acs_merged["zcta"].isin(target_zctas)].copy()
    used_target_file = True

filtered_acs["state_fips"] = STATE_FIPS

print(f"ACS ZCTAs before filtering: {acs_merged['zcta'].nunique():,}")
print(f"ACS rows after filtering: {len(filtered_acs):,}")
print(f"Filtered ZCTAs: {filtered_acs['zcta'].nunique():,}")

Using target ZCTA list from: data\processed\census\tx_zcta_to_puma_crosswalk_2020.csv
Inferred ZIP/ZCTA column: zcta
Target ZCTA count: 1,915
ACS ZCTAs before filtering: 33,772
ACS rows after filtering: 1,915
Filtered ZCTAs: 1,915


## 6. Calculate Target Proportions

The target categories map the detailed ACS variables into the simplified customer fields used by the postal demo schema.

In [6]:
def sum_columns(frame: pd.DataFrame, columns: list[str]) -> pd.Series:
    return frame[columns].sum(axis=1, min_count=1)


def safe_divide(numerator, denominator) -> pd.Series:
    numerator_series = pd.Series(numerator, index=denominator.index if hasattr(denominator, "index") else None, dtype="float64")
    denominator_series = pd.Series(denominator, index=numerator_series.index, dtype="float64")
    valid = denominator_series.notna() & denominator_series.gt(0)
    result = pd.Series(np.nan, index=numerator_series.index, dtype="float64")
    result.loc[valid] = numerator_series.loc[valid] / denominator_series.loc[valid]
    return result


def clip_pct_columns(frame: pd.DataFrame, pct_columns: list[str]) -> pd.DataFrame:
    clipped = frame.copy()
    clipped_any = pd.Series(False, index=clipped.index)
    for column in pct_columns:
        clipped_flag = clipped[column].notna() & ~clipped[column].between(0, 1)
        clipped[f"{column}_was_clipped"] = clipped_flag.astype(int)
        clipped_any = clipped_any | clipped_flag
        clipped[column] = clipped[column].clip(0, 1)
    clipped["any_pct_was_clipped"] = clipped_any.astype(int)
    return clipped


targets = filtered_acs.copy()

targets["income_total"] = targets["B19001_001E"]
targets["income_low_count"] = sum_columns(targets, [f"B19001_{i:03d}E" for i in range(2, 11)])
targets["income_medium_count"] = sum_columns(targets, ["B19001_011E", "B19001_012E", "B19001_013E"])
targets["income_high_count"] = sum_columns(targets, ["B19001_014E", "B19001_015E"])
targets["income_very_high_count"] = sum_columns(targets, ["B19001_016E", "B19001_017E"])

targets["tenure_total"] = targets["B25003_001E"]
targets["owner_count"] = targets["B25003_002E"]
targets["renter_count"] = targets["B25003_003E"]

targets["household_type_total"] = targets["B11001_001E"]
targets["married_household_count"] = targets["B11001_003E"]

targets["education_total"] = targets["B15003_001E"]
targets["partial_high_school_count"] = sum_columns(targets, [f"B15003_{i:03d}E" for i in range(2, 17)])
targets["high_school_count"] = sum_columns(targets, ["B15003_017E", "B15003_018E"])
targets["partial_college_count"] = sum_columns(targets, ["B15003_019E", "B15003_020E", "B15003_021E"])
targets["bachelors_count"] = targets["B15003_022E"]
targets["graduate_degree_count"] = sum_columns(targets, ["B15003_023E", "B15003_024E", "B15003_025E"])

age_18_24_columns = ["B01001_007E", "B01001_008E", "B01001_009E", "B01001_010E", "B01001_031E", "B01001_032E", "B01001_033E", "B01001_034E"]
age_25_34_columns = ["B01001_011E", "B01001_012E", "B01001_035E", "B01001_036E"]
age_35_44_columns = ["B01001_013E", "B01001_014E", "B01001_037E", "B01001_038E"]
age_45_64_columns = ["B01001_015E", "B01001_016E", "B01001_017E", "B01001_018E", "B01001_019E", "B01001_039E", "B01001_040E", "B01001_041E", "B01001_042E", "B01001_043E"]
age_65_plus_columns = ["B01001_020E", "B01001_021E", "B01001_022E", "B01001_023E", "B01001_024E", "B01001_025E", "B01001_044E", "B01001_045E", "B01001_046E", "B01001_047E", "B01001_048E", "B01001_049E"]
age_18_plus_columns = age_18_24_columns + age_25_34_columns + age_35_44_columns + age_45_64_columns + age_65_plus_columns

targets["age_18_plus_count"] = sum_columns(targets, age_18_plus_columns)
targets["age_18_24_count"] = sum_columns(targets, age_18_24_columns)
targets["age_25_34_count"] = sum_columns(targets, age_25_34_columns)
targets["age_35_44_count"] = sum_columns(targets, age_35_44_columns)
targets["age_45_64_count"] = sum_columns(targets, age_45_64_columns)
targets["age_65_plus_count"] = sum_columns(targets, age_65_plus_columns)

targets["income_low_pct"] = safe_divide(targets["income_low_count"], targets["income_total"])
targets["income_medium_pct"] = safe_divide(targets["income_medium_count"], targets["income_total"])
targets["income_high_pct"] = safe_divide(targets["income_high_count"], targets["income_total"])
targets["income_very_high_pct"] = safe_divide(targets["income_very_high_count"], targets["income_total"])
targets["owner_pct"] = safe_divide(targets["owner_count"], targets["tenure_total"])
targets["renter_pct"] = safe_divide(targets["renter_count"], targets["tenure_total"])
targets["married_household_pct"] = safe_divide(targets["married_household_count"], targets["household_type_total"])
targets["partial_high_school_pct"] = safe_divide(targets["partial_high_school_count"], targets["education_total"])
targets["high_school_pct"] = safe_divide(targets["high_school_count"], targets["education_total"])
targets["partial_college_pct"] = safe_divide(targets["partial_college_count"], targets["education_total"])
targets["bachelors_pct"] = safe_divide(targets["bachelors_count"], targets["education_total"])
targets["graduate_degree_pct"] = safe_divide(targets["graduate_degree_count"], targets["education_total"])
targets["age_18_24_pct"] = safe_divide(targets["age_18_24_count"], targets["age_18_plus_count"])
targets["age_25_34_pct"] = safe_divide(targets["age_25_34_count"], targets["age_18_plus_count"])
targets["age_35_44_pct"] = safe_divide(targets["age_35_44_count"], targets["age_18_plus_count"])
targets["age_45_64_pct"] = safe_divide(targets["age_45_64_count"], targets["age_18_plus_count"])
targets["age_65_plus_pct"] = safe_divide(targets["age_65_plus_count"], targets["age_18_plus_count"])

targets["total_households"] = targets["income_total"]
targets["source_year"] = ACS_YEAR
targets["source_dataset"] = ACS_DATASET

PCT_COLUMNS = [column for column in targets.columns if column.endswith("_pct")]
targets = clip_pct_columns(targets, PCT_COLUMNS)

print(f"Calculated target profiles: {len(targets):,} rows")

Calculated target profiles: 1,915 rows


## 7. Final Tables

The main CSV keeps the simple join-ready target schema. The audit CSV keeps counts, denominators, clipping flags, and raw ACS estimates.

In [7]:
FINAL_COLUMNS = [
    "zcta",
    "zcta_name",
    "state_fips",
    "total_households",
    "income_low_pct",
    "income_medium_pct",
    "income_high_pct",
    "income_very_high_pct",
    "owner_pct",
    "renter_pct",
    "married_household_pct",
    "partial_high_school_pct",
    "high_school_pct",
    "partial_college_pct",
    "bachelors_pct",
    "graduate_degree_pct",
    "age_18_24_pct",
    "age_25_34_pct",
    "age_35_44_pct",
    "age_45_64_pct",
    "age_65_plus_pct",
    "source_year",
    "source_dataset",
]

COUNT_AND_DENOMINATOR_COLUMNS = [
    "income_total",
    "income_low_count",
    "income_medium_count",
    "income_high_count",
    "income_very_high_count",
    "tenure_total",
    "owner_count",
    "renter_count",
    "household_type_total",
    "married_household_count",
    "education_total",
    "partial_high_school_count",
    "high_school_count",
    "partial_college_count",
    "bachelors_count",
    "graduate_degree_count",
    "age_18_plus_count",
    "age_18_24_count",
    "age_25_34_count",
    "age_35_44_count",
    "age_45_64_count",
    "age_65_plus_count",
]

CLIP_FLAG_COLUMNS = [column for column in targets.columns if column.endswith("_was_clipped") or column == "any_pct_was_clipped"]
RAW_ACS_COLUMNS = [column for column in ALL_ACS_VARIABLES if column in targets.columns]
AUDIT_COLUMNS = FINAL_COLUMNS + COUNT_AND_DENOMINATOR_COLUMNS + CLIP_FLAG_COLUMNS + RAW_ACS_COLUMNS

final_targets = targets[FINAL_COLUMNS].copy().sort_values("zcta").reset_index(drop=True)
audit_targets = targets[AUDIT_COLUMNS].copy().sort_values("zcta").reset_index(drop=True)

for column in PCT_COLUMNS:
    if column in final_targets.columns:
        final_targets[column] = final_targets[column].round(6)

for column in COUNT_AND_DENOMINATOR_COLUMNS + RAW_ACS_COLUMNS:
    if column in audit_targets.columns:
        audit_targets[column] = audit_targets[column].round().astype("Int64")

print(f"Final target rows: {len(final_targets):,}")
print(f"Audit columns: {len(audit_targets.columns):,}")

Final target rows: 1,915
Audit columns: 149


## 8. Validation Checks

The notebook fails on structural issues and warns for sparse ACS denominators. Small denominator problems can happen for low-population ZCTAs, so those are reported without stopping the pipeline.

In [8]:
def report_denominator_issues(frame: pd.DataFrame) -> pd.DataFrame:
    denominator_columns = ["income_total", "tenure_total", "household_type_total", "education_total", "age_18_plus_count"]
    issue_mask = pd.Series(False, index=frame.index)
    for column in denominator_columns:
        issue_mask = issue_mask | frame[column].isna() | frame[column].le(0)
    issues = frame.loc[issue_mask, ["zcta", "zcta_name", *denominator_columns]].copy()
    print(f"Rows with missing or zero denominators: {len(issues):,}")
    if not issues.empty:
        display(issues.head(25))
    return issues


def validate_group_sum(frame: pd.DataFrame, columns: list[str], label: str, tolerance: float = 0.01) -> None:
    group_sum = frame[columns].sum(axis=1, min_count=len(columns))
    bad = frame.loc[group_sum.notna() & ~np.isclose(group_sum, 1.0, atol=tolerance), ["zcta", "zcta_name", *columns]].copy()
    print(f"{label} group-sum warnings: {len(bad):,}")
    if not bad.empty:
        bad[f"{label}_sum"] = group_sum.loc[bad.index]
        display(bad.head(25))


def validate_targets(final_frame: pd.DataFrame, audit_frame: pd.DataFrame) -> None:
    assert list(final_frame.columns) == FINAL_COLUMNS, "Final columns are not in the requested order"
    assert final_frame["zcta"].astype("string").str.len().eq(5).all(), "zcta must always be 5 characters"
    assert not final_frame["zcta"].duplicated().any(), "zcta must not have duplicates"
    assert final_frame["state_fips"].eq(STATE_FIPS).all(), "state_fips must always be 48"
    assert pd.api.types.is_numeric_dtype(final_frame["total_households"]), "total_households must be numeric"
    assert final_frame["source_year"].eq(ACS_YEAR).all(), "source_year must be 2024"

    pct_columns = [column for column in final_frame.columns if column.endswith("_pct")]
    pct_ranges_ok = final_frame[pct_columns].apply(lambda series: series.dropna().between(0, 1).all()).all()
    assert pct_ranges_ok, "All pct columns must be between 0 and 1, ignoring nulls"

    report_denominator_issues(audit_frame)
    null_pct_rows = final_frame.loc[final_frame[pct_columns].isna().any(axis=1), ["zcta", "zcta_name", *pct_columns]]
    print(f"Rows with any null final pct columns: {len(null_pct_rows):,}")
    if not null_pct_rows.empty:
        display(null_pct_rows.head(25))

    validate_group_sum(final_frame, ["income_low_pct", "income_medium_pct", "income_high_pct", "income_very_high_pct"], "income")
    validate_group_sum(final_frame, ["owner_pct", "renter_pct"], "tenure")
    validate_group_sum(final_frame, ["partial_high_school_pct", "high_school_pct", "partial_college_pct", "bachelors_pct", "graduate_degree_pct"], "education")
    validate_group_sum(final_frame, ["age_18_24_pct", "age_25_34_pct", "age_35_44_pct", "age_45_64_pct", "age_65_plus_pct"], "age")

    print("Structural validation complete.")


validate_targets(final_targets, audit_targets)

if used_target_file:
    matched_zctas = set(final_targets["zcta"])
    missing_zctas = sorted(target_zctas - matched_zctas)
    print(f"Target ZCTAs: {len(target_zctas):,}")
    print(f"Matched in ACS: {len(matched_zctas):,}")
    print(f"Missing from ACS: {len(missing_zctas):,}")
    missing_frame = pd.DataFrame({"zcta": missing_zctas})
    missing_frame.to_csv(OUTPUT_MISSING_ZCTAS, index=False)
    if missing_zctas:
        display(missing_frame.head(50))
        print(f"Saved missing target ZCTAs: {OUTPUT_MISSING_ZCTAS}")

Rows with missing or zero denominators: 27


,zcta,zcta_name,income_total,tenure_total,household_type_total,education_total,age_18_plus_count
309,75711,ZCTA5 75711,0,0,0,0,156
365,75884,ZCTA5 75884,0,0,0,2492,2697
366,75886,ZCTA5 75886,0,0,0,2349,2456
490,76129,ZCTA5 76129,0,0,0,74,1631
506,76203,ZCTA5 76203,0,0,0,0,741
575,76402,ZCTA5 76402,0,0,0,0,731
662,76596,ZCTA5 76596,0,0,0,971,1060
663,76597,ZCTA5 76597,0,0,0,2856,3005
664,76598,ZCTA5 76598,0,0,0,480,491
665,76599,ZCTA5 76599,0,0,0,1903,1980


Rows with any null final pct columns: 27


,zcta,zcta_name,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,married_household_pct,partial_high_school_pct,high_school_pct,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct
309,75711,ZCTA5 75711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000,0.000000,0.000000
365,75884,ZCTA5 75884,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.286918,0.386838,0.293740,0.020465,0.012039,0.076010,0.166852,0.282907,0.444939,0.029292
366,75886,ZCTA5 75886,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.290762,0.437633,0.247339,0.022563,0.001703,0.043567,0.207655,0.322883,0.410423,0.015472
490,76129,ZCTA5 76129,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.148649,0.689189,0.162162,0.954629,0.038627,0.006744,0.000000,0.000000
506,76203,ZCTA5 76203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000,0.000000,0.000000
575,76402,ZCTA5 76402,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000,0.000000,0.000000
662,76596,ZCTA5 76596,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.171988,0.325438,0.473738,0.028836,0.000000,0.083962,0.258491,0.355660,0.273585,0.028302
663,76597,ZCTA5 76597,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.302171,0.468487,0.184174,0.017157,0.028011,0.049584,0.310815,0.290516,0.332779,0.016306
664,76598,ZCTA5 76598,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.114583,0.460417,0.425000,0.000000,0.000000,0.022403,0.364562,0.228106,0.352342,0.032587
665,76599,ZCTA5 76599,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.190751,0.450867,0.323699,0.024172,0.010510,0.038889,0.311616,0.425758,0.203535,0.020202


income group-sum warnings: 0
tenure group-sum warnings: 0
education group-sum warnings: 0
age group-sum warnings: 0
Structural validation complete.
Target ZCTAs: 1,915
Matched in ACS: 1,915
Missing from ACS: 0


## 9. Save Outputs

In [9]:
def save_outputs(final_frame: pd.DataFrame, audit_frame: pd.DataFrame) -> None:
    final_frame.to_csv(OUTPUT_MAIN, index=False)
    audit_frame.to_csv(OUTPUT_AUDIT, index=False)
    print(f"Saved main ACS targets: {OUTPUT_MAIN}")
    print(f"Saved audit ACS targets: {OUTPUT_AUDIT}")


save_outputs(final_targets, audit_targets)

Saved main ACS targets: processed\census\tx_zcta_acs_targets_2024.csv
Saved audit ACS targets: processed\census\tx_zcta_acs_targets_2024_audit.csv


## 10. Examples and Summary Statistics

In [10]:
display(final_targets.head(10))

,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
0,73960,ZCTA5 73960,48,5,0.400000,0.600000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.600000,0.000000,0.000000,0.000000,0.000000,0.600000,0.400000,2024,acs/acs5
1,75001,ZCTA5 75001,48,10579,0.236790,0.450610,0.179412,0.133188,0.162870,0.837130,...,0.223223,0.396776,0.236875,0.110228,0.355560,0.188538,0.238095,0.107579,2024,acs/acs5
2,75002,ZCTA5 75002,48,23656,0.142205,0.242053,0.165962,0.449780,0.784410,0.215590,...,0.276636,0.327434,0.197031,0.126074,0.131135,0.177717,0.401281,0.163793,2024,acs/acs5
3,75006,ZCTA5 75006,48,18774,0.248535,0.357569,0.208533,0.185363,0.515181,0.484819,...,0.256585,0.238975,0.110818,0.119708,0.202340,0.181081,0.310349,0.186521,2024,acs/acs5
4,75007,ZCTA5 75007,48,20788,0.196123,0.235424,0.217481,0.350972,0.673562,0.326438,...,0.288116,0.276265,0.163206,0.095334,0.168130,0.206214,0.350845,0.179477,2024,acs/acs5
5,75009,ZCTA5 75009,48,10605,0.140311,0.118340,0.158699,0.582650,0.920132,0.079868,...,0.230266,0.441184,0.192479,0.081243,0.171041,0.277029,0.280659,0.190027,2024,acs/acs5
6,75010,ZCTA5 75010,48,13516,0.166025,0.284034,0.234093,0.315848,0.516721,0.483279,...,0.244183,0.378895,0.222108,0.082142,0.221371,0.228955,0.340112,0.127420,2024,acs/acs5
7,75013,ZCTA5 75013,48,18064,0.144708,0.181577,0.185839,0.487876,0.636238,0.363762,...,0.212786,0.366347,0.299918,0.090262,0.165195,0.212574,0.387132,0.144837,2024,acs/acs5
8,75019,ZCTA5 75019,48,16954,0.113896,0.204495,0.204318,0.477291,0.633243,0.366757,...,0.182653,0.384916,0.310510,0.081725,0.147942,0.203981,0.404710,0.161643,2024,acs/acs5
9,75020,ZCTA5 75020,48,10085,0.346158,0.358156,0.158354,0.137333,0.587407,0.412593,...,0.384217,0.128588,0.067659,0.100648,0.129573,0.160587,0.329584,0.279608,2024,acs/acs5


In [11]:
example_zctas = ["77494", "77002", "75201", "78701", "79936"]
for example_zcta in example_zctas:
    rows = final_targets.loc[final_targets["zcta"].eq(example_zcta)]
    if rows.empty:
        print(f"ZCTA {example_zcta}: not present in final targets.")
    else:
        print(f"ZCTA {example_zcta}")
        display(rows)

ZCTA 77494


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
1013,77494,ZCTA5 77494,48,42924,0.151896,0.16471,0.187308,0.496086,0.708625,0.291375,...,0.191662,0.383672,0.283385,0.094167,0.107261,0.26907,0.396565,0.132936,2024,acs/acs5


ZCTA 77002


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
798,77002,ZCTA5 77002,48,7644,0.223312,0.346415,0.129121,0.301151,0.082287,0.917713,...,0.213983,0.232819,0.155212,0.128125,0.346353,0.266016,0.222962,0.036544,2024,acs/acs5


ZCTA 75201


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
114,75201,ZCTA5 75201,48,12525,0.196647,0.269461,0.231537,0.302355,0.106028,0.893972,...,0.15981,0.402817,0.296733,0.137305,0.429801,0.175835,0.177129,0.07993,2024,acs/acs5


ZCTA 78701


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
1508,78701,ZCTA5 78701,48,7949,0.163794,0.087181,0.167946,0.581079,0.453013,0.546987,...,0.103508,0.470334,0.353141,0.067569,0.296908,0.178984,0.278924,0.177614,2024,acs/acs5


ZCTA 79936


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
1913,79936,ZCTA5 79936,48,35305,0.339867,0.360884,0.165954,0.133296,0.705028,0.294972,...,0.336523,0.191058,0.086391,0.143442,0.199854,0.150032,0.326074,0.180598,2024,acs/acs5


In [12]:
summary_stats = pd.DataFrame(
    {
        "metric": ["number of ZCTAs", "mean total_households", "median total_households"],
        "value": [
            final_targets["zcta"].nunique(),
            round(final_targets["total_households"].mean(), 2),
            round(final_targets["total_households"].median(), 2),
        ],
    }
)
display(summary_stats)

print("Top 10 ZCTAs by total_households")
display(final_targets.sort_values("total_households", ascending=False).head(10))

print("Top 10 ZCTAs by income_very_high_pct")
display(final_targets.sort_values("income_very_high_pct", ascending=False).head(10))

print("Top 10 ZCTAs by owner_pct")
display(final_targets.sort_values("owner_pct", ascending=False).head(10))

print("Top 10 ZCTAs by graduate_degree_pct")
display(final_targets.sort_values("graduate_degree_pct", ascending=False).head(10))

,metric,value
0,number of ZCTAs,1915.0
1,mean total_households,5740.2
2,median total_households,2181.0


Top 10 ZCTAs by total_households


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
1492,78660,ZCTA5 78660,48,45630,0.155271,0.278326,0.244620,0.321784,0.661144,0.338856,...,0.293099,0.295262,0.156304,0.089988,0.202713,0.273345,0.334792,0.099162,2024,acs/acs5
1013,77494,ZCTA5 77494,48,42924,0.151896,0.164710,0.187308,0.496086,0.708625,0.291375,...,0.191662,0.383672,0.283385,0.094167,0.107261,0.269070,0.396565,0.132936,2024,acs/acs5
978,77449,ZCTA5 77449,48,40918,0.221907,0.357129,0.246151,0.174813,0.682878,0.317122,...,0.318269,0.193968,0.081824,0.115332,0.209165,0.233468,0.331309,0.110726,2024,acs/acs5
1244,78130,ZCTA5 78130,48,40747,0.259995,0.321545,0.206126,0.212335,0.658601,0.341399,...,0.308207,0.264157,0.098836,0.113994,0.210007,0.190179,0.303319,0.182501,2024,acs/acs5
879,77084,ZCTA5 77084,48,36886,0.259665,0.361275,0.216369,0.162690,0.600146,0.399854,...,0.298585,0.204447,0.098295,0.131755,0.204195,0.203256,0.318176,0.142619,2024,acs/acs5
1057,77573,ZCTA5 77573,48,35988,0.167056,0.239246,0.190758,0.402940,0.757503,0.242497,...,0.309283,0.337203,0.152969,0.098298,0.151375,0.227486,0.343746,0.179095,2024,acs/acs5
1478,78641,ZCTA5 78641,48,35410,0.134312,0.206439,0.219458,0.439791,0.771844,0.228156,...,0.284887,0.311478,0.200695,0.089375,0.180917,0.254230,0.343018,0.132460,2024,acs/acs5
1913,79936,ZCTA5 79936,48,35305,0.339867,0.360884,0.165954,0.133296,0.705028,0.294972,...,0.336523,0.191058,0.086391,0.143442,0.199854,0.150032,0.326074,0.180598,2024,acs/acs5
965,77433,ZCTA5 77433,48,34982,0.153279,0.182808,0.195129,0.468784,0.814505,0.185495,...,0.243282,0.311387,0.204638,0.101678,0.171361,0.252129,0.343079,0.131753,2024,acs/acs5
1498,78666,ZCTA5 78666,48,33949,0.447789,0.321158,0.132876,0.098177,0.380777,0.619223,...,0.303533,0.220412,0.136101,0.358804,0.193414,0.130499,0.177113,0.140169,2024,acs/acs5


Top 10 ZCTAs by income_very_high_pct


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
283,75659,ZCTA5 75659,48,18,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.486486,0.513514,2024,acs/acs5
1691,79231,ZCTA5 79231,48,15,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.625000,0.000000,0.0000,0.0,0.0,0.0,1.000000,0.000000,2024,acs/acs5
1249,78144,ZCTA5 78144,48,29,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.617021,0.382979,2024,acs/acs5
215,75475,ZCTA5 75475,48,52,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.400000,0.000000,0.6000,0.0,0.6,0.0,0.400000,0.000000,2024,acs/acs5
711,76680,ZCTA5 76680,48,57,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.000000,1.000000,0.0000,0.0,1.0,0.0,0.000000,0.000000,2024,acs/acs5
1712,79259,ZCTA5 79259,48,5,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.000000,0.814815,0.0000,0.0,0.0,0.0,1.000000,0.000000,2024,acs/acs5
796,76957,ZCTA5 76957,48,11,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.000000,0.000000,1.0000,0.0,0.0,1.0,0.000000,0.000000,2024,acs/acs5
1685,79223,ZCTA5 79223,48,11,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.500000,0.233333,0.0000,0.0,0.0,0.5,0.000000,0.500000,2024,acs/acs5
1160,77976,ZCTA5 77976,48,9,0.00000,0.0,0.0,1.00000,1.0,0.0,...,0.437500,0.000000,0.5625,0.0,1.0,0.0,0.000000,0.000000,2024,acs/acs5
748,76848,ZCTA5 76848,48,32,0.21875,0.0,0.0,0.78125,1.0,0.0,...,0.883333,0.116667,0.0000,0.0,0.0,0.0,0.000000,1.000000,2024,acs/acs5


Top 10 ZCTAs by owner_pct


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
0,73960,ZCTA5 73960,48,5,0.400000,0.600000,0.000000,0.000000,1.0,0.0,...,0.000000,0.600000,0.000000,0.000000,0.000000,0.000000,0.600000,0.400000,2024,acs/acs5
185,75434,ZCTA5 75434,48,6,1.000000,0.000000,0.000000,0.000000,1.0,0.0,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,2024,acs/acs5
1197,78027,ZCTA5 78027,48,155,0.129032,0.122581,0.703226,0.045161,1.0,0.0,...,0.685535,0.223270,0.022013,0.000000,0.308176,0.000000,0.440252,0.251572,2024,acs/acs5
1191,78021,ZCTA5 78021,48,51,0.000000,0.568627,0.000000,0.431373,1.0,0.0,...,0.493671,0.367089,0.050633,0.000000,0.177215,0.367089,0.088608,0.367089,2024,acs/acs5
215,75475,ZCTA5 75475,48,52,0.000000,0.000000,0.000000,1.000000,1.0,0.0,...,0.400000,0.000000,0.600000,0.000000,0.600000,0.000000,0.400000,0.000000,2024,acs/acs5
1164,77982,ZCTA5 77982,48,288,0.263889,0.392361,0.125000,0.218750,1.0,0.0,...,0.319626,0.209346,0.205607,0.018349,0.000000,0.196330,0.091743,0.693578,2024,acs/acs5
1161,77977,ZCTA5 77977,48,194,0.603093,0.216495,0.000000,0.180412,1.0,0.0,...,0.319088,0.000000,0.000000,0.011268,0.101408,0.447887,0.329577,0.109859,2024,acs/acs5
1160,77976,ZCTA5 77976,48,9,0.000000,0.000000,0.000000,1.000000,1.0,0.0,...,0.437500,0.000000,0.562500,0.000000,1.000000,0.000000,0.000000,0.000000,2024,acs/acs5
1157,77971,ZCTA5 77971,48,221,0.425339,0.321267,0.000000,0.253394,1.0,0.0,...,0.411429,0.000000,0.000000,0.180967,0.000000,0.000000,0.333853,0.485179,2024,acs/acs5
1150,77961,ZCTA5 77961,48,58,1.000000,0.000000,0.000000,0.000000,1.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2024,acs/acs5


Top 10 ZCTAs by graduate_degree_pct


,zcta,zcta_name,state_fips,total_households,income_low_pct,income_medium_pct,income_high_pct,income_very_high_pct,owner_pct,renter_pct,...,partial_college_pct,bachelors_pct,graduate_degree_pct,age_18_24_pct,age_25_34_pct,age_35_44_pct,age_45_64_pct,age_65_plus_pct,source_year,source_dataset
796,76957,ZCTA5 76957,48,11,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,2024,acs/acs5
1139,77876,ZCTA5 77876,48,24,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.058824,0.941176,0.000000,0.000000,0.000000,0.000000,1.000000,2024,acs/acs5
787,76937,ZCTA5 76937,48,96,0.000000,0.260417,0.697917,0.041667,0.802083,0.197917,...,0.000000,0.068323,0.732919,0.000000,0.118012,0.732919,0.049689,0.099379,2024,acs/acs5
1335,78347,ZCTA5 78347,48,60,0.000000,0.000000,0.433333,0.566667,1.000000,0.000000,...,0.276596,0.000000,0.723404,0.000000,0.000000,0.723404,0.000000,0.276596,2024,acs/acs5
215,75475,ZCTA5 75475,48,52,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,...,0.400000,0.000000,0.600000,0.000000,0.600000,0.000000,0.400000,0.000000,2024,acs/acs5
1122,77843,ZCTA5 77843,48,0,NaN,NaN,NaN,NaN,NaN,NaN,...,0.098214,0.330357,0.571429,0.986300,0.011131,0.002569,0.000000,0.000000,2024,acs/acs5
1160,77976,ZCTA5 77976,48,9,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,...,0.437500,0.000000,0.562500,0.000000,1.000000,0.000000,0.000000,0.000000,2024,acs/acs5
801,77005,ZCTA5 77005,48,10022,0.129216,0.120934,0.117841,0.632010,0.718918,0.281082,...,0.074999,0.322272,0.556307,0.225239,0.108666,0.160085,0.299837,0.206172,2024,acs/acs5
977,77448,ZCTA5 77448,48,58,0.568966,0.431034,0.000000,0.000000,1.000000,0.000000,...,0.070175,0.000000,0.538012,0.000000,0.035088,0.076023,0.000000,0.888889,2024,acs/acs5
826,77030,ZCTA5 77030,48,6714,0.268543,0.327376,0.125707,0.278374,0.340632,0.659368,...,0.141741,0.292348,0.507335,0.152201,0.301897,0.178329,0.205709,0.161865,2024,acs/acs5


## Final Interpretation

Each row represents one Texas ZCTA. The proportion columns are demographic targets for later synthetic customer generation. For example, if ZCTA 77494 has 1,000 synthetic postal customers and `owner_pct = 0.78`, then the later demographic sampler should aim for roughly 780 homeowners among customers assigned to that ZIP.

These ACS targets should be used to control ZIP-level distributions, while ACS PUMS donor records will later preserve realistic combinations of age, education, income, household structure, occupation, and homeownership.